# Clase **Estudiante** : Sistema de gestion Academica

https://gist.github.com/robintux/4d332388e3ed77f5ecf63fa7fd4ade36

In [ ]:
class Estudiante:
    """Representa a un estudiante con nombre, edad y calificaciones."""

    def __init__(self, nombre, edad):
        """Inicializa un estudiante con nombre, edad y una lista vacía de calificaciones."""
        self.nombre = nombre                 # atributo de instancia
        self.edad = edad                     # atributo de instancia
        self._calificaciones = []            # atributo protegido (convención)

    def agregar_calificacion(self, nota):
        """Agrega una calificación válida (0-10) a la lista."""
        if 0 <= nota <= 10:
            self._calificaciones.append(nota)
        else:
            raise ValueError("La calificación debe estar entre 0 y 10.")

    @property
    def promedio(self):
        """Calcula y devuelve el promedio de las calificaciones."""
        if not self._calificaciones:
            return 0.0
        return sum(self._calificaciones) / len(self._calificaciones)


## Analisis de esta primera implementacion

### Aspectos Positivos
- Encapsulamiento basico : Uso de `_calificaciones` como atributo protegido
- Validacion de entrada : Control de rango valido de calificaciones
- propiedad `promedio` : uso correcto del decorador `@property` para calculo *on-demand*
- Claridad y legibilidad : Codigo limpio y docstrings.

### Puntos de Mejora
- Falta de inmutabilidad controlada y validacion del constructor
  - No se valida que `nombre` sea un str no vacio y que `edad` sea un entero positivo
  - Esto puede llevar a inconsistencias en produccion

- Diseño monolitico sin posibilidad de extension
  - No permite diferentes esquemas de calificacion
  - No contempla metadatos de calificaciones (fecha, tipo de evaluacion, asignatura, etc)

- Calculo del promedio : simplista y no alineado a buenas practicas estadisticas
  - Devuelve 0 si no hay calificaciones. Esto puede sesgar analisis estadisticos
  - Idealmente deberia devolver None o lanzar una excepcion explicita.

- Falta de soporte para serializacion y persistencia
  - No es facil de integrar con BD o API's (JSON, XML)
  - No implementa metodos estandar como `__repr__` , `__str__`, o compatibilidad con `dataclasses/pydantic`

- Ausencia de tipado estatico

- No se consideran casos de uso futuros.

### Propuesta de mejora integral
- Tipado estatico (typing)
- Validacion robusta (logica interna o `pydantic`)
- Extensibilidad (Estructura preparada para metadatos)
- Mejor formulacion de la parte estadistica
- Serializacion (`__repr__`, `to_dict`, etc)
- Preparacion para persistencia y microservicios.


# Version mejorada del sistema de gestion academica

https://gist.github.com/robintux/c8eb73efd51b9d6c10d9233884843cea



In [2]:
from __future__ import annotations
from typing import List, Optional, Union
from math import nan
import json


class Calificacion:
    """Representa una calificación con metadatos (extensible)."""
    def __init__(self, valor: float, asignatura: Optional[str] = None, fecha: Optional[str] = None):
        if not (0 <= valor <= 10):
            raise ValueError("La calificación debe estar entre 0 y 10.")
        if not isinstance(valor, (int, float)):
            raise TypeError("La calificación debe ser numérica.")
        self.valor = float(valor)
        self.asignatura = asignatura
        self.fecha = fecha

    """
    Futuras extensiones de la clase Calificacion
      Validar el formato de fecha
      Agregar el tipo de evaluacion
      Soportar ponderacion
      Integrar con ontologias educativas
      ...
    """

    def __repr__(self):
        return f"Calificacion(valor={self.valor}, asignatura={self.asignatura})"


class Estudiante:
    """Representa a un estudiante con nombre, edad y calificaciones estructuradas."""

    def __init__(self, nombre: str, edad: int):
        if not isinstance(nombre, str) or not nombre.strip():
            raise ValueError("El nombre debe ser una cadena no vacía.")
        if not isinstance(edad, int) or edad < 0:
            raise ValueError("La edad debe ser un entero no negativo.")

        self.nombre = nombre.strip()
        self.edad = edad
        self._calificaciones: List[Calificacion] = []

    def agregar_calificacion(
        self,
        valor: Union[int, float],
        asignatura: Optional[str] = None,
        fecha: Optional[str] = None
    ) -> None:
        """Agrega una calificación con metadatos opcionales."""
        calif = Calificacion(valor, asignatura, fecha)
        self._calificaciones.append(calif)

    def agregar_calificaciones(self, notas: List[Union[int, float]]) -> None:
        """Agrega múltiples calificaciones sin metadatos (modo rápido)."""
        for nota in notas:
            self.agregar_calificacion(nota)

    @property
    def promedio(self) -> float:
        """Devuelve el promedio aritmético. Si no hay calificaciones, devuelve NaN."""
        if not self._calificaciones:
            return nan
        return sum(c.valor for c in self._calificaciones) / len(self._calificaciones)

    @property
    def calificaciones_valores(self) -> List[float]:
        """Devuelve solo los valores numéricos de las calificaciones."""
        return [c.valor for c in self._calificaciones]

    def estadisticas_descriptivas(self) -> dict:
        """Devuelve estadísticas básicas: promedio, mediana, desviación estándar."""
        from statistics import mean, median, stdev
        valores = self.calificaciones_valores
        if not valores:
            return {"promedio": nan, "mediana": nan, "desviacion_estandar": nan}
        return {
            "promedio": mean(valores),
            "mediana": median(valores),
            "desviacion_estandar": stdev(valores) if len(valores) > 1 else 0.0
        }

    def to_dict(self) -> dict:
        """Serializa el estudiante a un diccionario (compatible con JSON)."""
        return {
            "nombre": self.nombre,
            "edad": self.edad,
            "calificaciones": [
                {
                    "valor": c.valor,
                    "asignatura": c.asignatura,
                    "fecha": c.fecha
                }
                for c in self._calificaciones
            ],
            "promedio": self.promedio  # nan no es JSON serializable; manejar en capa superior
        }

    def __repr__(self):
        return f"Estudiante(nombre='{self.nombre}', edad={self.edad}, n_calif={len(self._calificaciones)})"

    def __str__(self):
        return f"{self.nombre} ({self.edad} años) - Promedio: {self.promedio:.2f} (n={len(self._calificaciones)})"

# Ejemplos de uso de la version mejorada


In [6]:
# Ejemplito1 : Creacion basica de un estudiantes y registro de calificaciones simples

# Crear un estudiante
estudiante1 = Estudiante("Pierina Floreano", 18)

# Agregar calificaciones (sin metadatos - modo rapido)
estudiante1.agregar_calificaciones([8, 8.5, 7.5])

print(estudiante1)

Pierina Floreano (18 años) - Promedio: 8.00 (n=3)


In [7]:
estudiante1

Estudiante(nombre='Pierina Floreano', edad=18, n_calif=3)

In [9]:
# Ejemplito2 : Registro de calificaciones con metadatos (asignatura y fecha)

estudiante2 = Estudiante("Madai Arellano", 17)

# Agregar calificaciones con contexto academico
estudiante2.agregar_calificacion(valor=9.2,asignatura="Matematica",fecha = "2025-03-26")
estudiante2.agregar_calificacion(valor=7.8,asignatura="Historia",fecha = "2025-06-26")
estudiante2.agregar_calificacion(valor=2.8,asignatura="Lenguaje y Redaccion",fecha = "2025-10-26")

print(estudiante2)

Madai Arellano (17 años) - Promedio: 6.60 (n=3)


In [10]:
# Acceder a valores crudos
estudiante2.calificaciones_valores

[9.2, 7.8, 2.8]

In [14]:
# Ejemplito3 : Validacion automatica (manejo de errores y excepcion)

try:
  # Intentar crear un estudiante con nombre vacio
  Estudiante("", 19)
except ValueError as e :
  print("Error ", e)

try:
  # Intentar agregar una calificacion invalida
  est = Estudiante("Lucia Araujo", 9)
  est.agregar_calificacion(12.0)
except ValueError as e:
  print("Error : ", e)

try:
  # Intento de calificacion con un tipo no numerico
  est.agregar_calificacion("ocho")
except TypeError as e:
  print("Error : ", e)

Error  El nombre debe ser una cadena no vacía.
Error :  La calificación debe estar entre 0 y 10.
Error :  '<=' not supported between instances of 'int' and 'str'


In [15]:
# Ejemplito4 : Estadisticas Descriptivas
estudiante4 = Estudiante("Ramiro Priale", 23)
estudiante4.agregar_calificaciones([6.0,7.5, 8, 9.1,9.5])


stats = estudiante4.estadisticas_descriptivas()
print(stats)

{'promedio': 8.02, 'mediana': 8.0, 'desviacion_estandar': 1.3881642554107203}


In [16]:
# Ejemplito5: manejo de un estudiante sin calificaciones
estudiante5 = Estudiante("Rosa Chauca", 13)

print("Promedio ", estudiante5.promedio)
print("Estadisticas ", estudiante5.estadisticas_descriptivas())
print(estudiante5)

Promedio  nan
Estadisticas  {'promedio': nan, 'mediana': nan, 'desviacion_estandar': nan}
Rosa Chauca (13 años) - Promedio: nan (n=0)


In [17]:
# Ejemplito6 : Serializacion a diccionario (para alimentar APIs o bases de datos )
estudiante6 = Estudiante("Felix Carrion", 16)
estudiante6.agregar_calificacion(9, "Fisica","2025-05-01")
estudiante6.agregar_calificacion(2, "Quimica","2025-08-01")

data = estudiante6.to_dict()
print(json.dumps(data, indent =2 , ensure_ascii=False))


{
  "nombre": "Felix Carrion",
  "edad": 16,
  "calificaciones": [
    {
      "valor": 9.0,
      "asignatura": "Fisica",
      "fecha": "2025-05-01"
    },
    {
      "valor": 2.0,
      "asignatura": "Quimica",
      "fecha": "2025-08-01"
    }
  ],
  "promedio": 5.5
}


## Justificacion de mejoras de la version mejorada

- Mejoras de escalabilidad y debugging a la clase `Calificacion`
- Validacion Estricta
- Uso de `nan`
- Mejoras en la implementacion del calculo de las estadisticas descriptivas.
- Mejoras a la serializacion (`to_dict`)
- Estandatizacion (dataclasses/pydantic/mypy)

# Herencia

La herencia es uno de los pilares fundamentales de la POO. Permite crear clases basadas en clases existentes, reutilizando, extendiendo y especializando su comportamiento y estado.

BUsquemos una jerarquia de clases alrededor de `Estudiante` y `Calificacion`, que ilistra de de forma clara y practica la importancia de la herencia, manteniendo la logica original de nuestras clases ya implementadas como base central del sistema.



## Objetivo

Diseñar un modelo de dominio academico mas rico, que permita presentar:

- Diferentes tipos de estudiantes (por ejemplo, de secundaria, universitarios, becados)
- Diferentes tipos de calificaciones (por ejemplo,
parciales, finales, trabajos practicos, ponderadas)
- Calculos de promedio especificos segun el tipo de estudiante o regimen academico.


## Jerarquia propuesta

```
Calificacion (base)
|--- CalificacionSimple (hereda de Calificacion)
|--- CalificacionPonderada (hereda de Calificacion)
|--- CalificacionFinal (hereda de CalificacionPonderada)

Estudiante (base)
|--- EstudianteSecundaria (hereda de Estudiante)
|--- EstudianteUniversitario (Hereda de Estudiante)
    |--- EstudianteBecado (hereda de EstudianteUniversitario)
    |--- EstudianteIntercambio (hereda de EstudianteUniversitario)
|--- EstudianteDistancia (hereda de Estudiante)

```



## Implementacion paso a paso


### Clase base : `Calificacion`

In [29]:
# Clase base : Ahora servira como clase base abstracta o concreta para tipos mas especificos

class Calificacion:
    """Representa una calificación con metadatos (extensible)."""
    def __init__(self, valor: float, asignatura: Optional[str] = None, fecha: Optional[str] = None):
        if not (0 <= valor <= 10):
            raise ValueError("La calificación debe estar entre 0 y 10.")
        if not isinstance(valor, (int, float)):
            raise TypeError("La calificación debe ser numérica.")
        self.valor = float(valor)
        self.asignatura = asignatura
        self.fecha = fecha

    """
    Futuras extensiones de la clase Calificacion
      Validar el formato de fecha
      Agregar el tipo de evaluacion
      Soportar ponderacion
      Integrar con ontologias educativas
      ...
    """

    def __repr__(self):
        return f"Calificacion(valor={self.valor}, asignatura={self.asignatura})"


### Clases Derivadas de `Calificacion`

#### a) CalificacionPonderada

Extiende a la clase `Calificacion` para incluir un peso, util para promedio ponderados.

In [48]:
class CalificacionPonderada(Calificacion):
  """
  Calificacion por ponderacion (ej: pc1 vale un 30% del total)
  """
  def __init__(self,
               valor: float,
               peso: float,
               asignatura: Optional[str] = None,
               fecha: Optional[str] = None):
    # Reutilizar la implementacion de Calificacion
    super().__init__(valor, asignatura, fecha)
    # En vista que el parametros peso no es analizado por la clase padre (Calificacion)
    # Validemos el valor del parametro peso
    if not (0 <= peso <= 1):
      raise ValueError("El peso debe estar en [0,1]")
    self.peso = peso

    def __repr__(self):
      return f"CalificacionPonderada(valor = {self.valor}, peso = {self.peso}, asignatura = {self.asignatura})"

# calificacion1 = CalificacionPonderada(5,0.5)
# calificacion1.valor

5.0

#### b) CalificacionFInal

Extiende `CalificacionPonderada` para indicar que es una evaluacion final (examen final, tesis, etc )

In [41]:
class CalificacionFinal(CalificacionPonderada):
  """
  Calificacion de un examen final o trabajo final , con
  aprobacion automatica.
  """

  def __init__(self,
               valor: float,
               peso: float ,
               asignatura: str,
               fecha: Optional[str] = None,
               aprobatorio: bool = True
               ):
    super().__init__(valor, peso, asignatura, fecha )
    self.aprobatorio = aprobatorio

  def es_aprobado(self) -> bool:
    return self.valor >= 8.5 and self.aprobatorio

  def __repr_(self):
    return f"CalificacionFinal(valor = {self.valor}, peso = {self.peso}, aprobado = {self.es_aprobado()})"



### Clase base : Estudiante


In [32]:

class Estudiante:
    """Representa a un estudiante con nombre, edad y calificaciones estructuradas."""

    def __init__(self, nombre: str, edad: int):
        if not isinstance(nombre, str) or not nombre.strip():
            raise ValueError("El nombre debe ser una cadena no vacía.")
        if not isinstance(edad, int) or edad < 0:
            raise ValueError("La edad debe ser un entero no negativo.")

        self.nombre = nombre.strip()
        self.edad = edad
        self._calificaciones: List[Calificacion] = []

    def agregar_calificacion(
        self,
        valor: Union[int, float],
        asignatura: Optional[str] = None,
        fecha: Optional[str] = None
    ) -> None:
        """Agrega una calificación con metadatos opcionales."""
        calif = Calificacion(valor, asignatura, fecha)
        self._calificaciones.append(calif)

    def agregar_calificaciones(self, notas: List[Union[int, float]]) -> None:
        """Agrega múltiples calificaciones sin metadatos (modo rápido)."""
        for nota in notas:
            self.agregar_calificacion(nota)

    @property
    def promedio(self) -> float:
        """Devuelve el promedio aritmético. Si no hay calificaciones, devuelve NaN."""
        if not self._calificaciones:
            return nan
        return sum(c.valor for c in self._calificaciones) / len(self._calificaciones)

    @property
    def calificaciones_valores(self) -> List[float]:
        """Devuelve solo los valores numéricos de las calificaciones."""
        return [c.valor for c in self._calificaciones]

    def estadisticas_descriptivas(self) -> dict:
        """Devuelve estadísticas básicas: promedio, mediana, desviación estándar."""
        from statistics import mean, median, stdev
        valores = self.calificaciones_valores
        if not valores:
            return {"promedio": nan, "mediana": nan, "desviacion_estandar": nan}
        return {
            "promedio": mean(valores),
            "mediana": median(valores),
            "desviacion_estandar": stdev(valores) if len(valores) > 1 else 0.0
        }

    def to_dict(self) -> dict:
        """Serializa el estudiante a un diccionario (compatible con JSON)."""
        return {
            "nombre": self.nombre,
            "edad": self.edad,
            "calificaciones": [
                {
                    "valor": c.valor,
                    "asignatura": c.asignatura,
                    "fecha": c.fecha
                }
                for c in self._calificaciones
            ],
            "promedio": self.promedio  # nan no es JSON serializable; manejar en capa superior
        }

    def __repr__(self):
        return f"Estudiante(nombre='{self.nombre}', edad={self.edad}, n_calif={len(self._calificaciones)})"

    def __str__(self):
        return f"{self.nombre} ({self.edad} años) - Promedio: {self.promedio:.2f} (n={len(self._calificaciones)})"

#### a) EstudianteSecundaria

Puede tener un sistema de promocion o regularidad diferente

In [33]:
class EstudianteSecundaria(Estudiante):
  """
  Estudiante de secundaria con logica de promocion simple
  """

  # No voy a modificar el constructor de la clase padre : Estudiante

  def promociona(self) -> bool :
    return self.promedio >= 6.0 and all(c.valor >=4 for c in self._calificaciones)

  def __repr__(self):
    if self.promociona():
      estado = "Promociona"
    else:
      estado = "No Promociona"
    return f"EstudianteSecundaria(nombre = {self.nombre}, edad = {self.edad}, estado = {estado})"



#### b) EstudianteUniversitario

Especializa el calculo de promedio para usar promedio ponderado si hay CalificacionPonderada

In [34]:
class EstudianteUniversitario(Estudiante):
  """
  Estudiante universitario que puede tener calificaciones ponderadas.
  """
  @property
  def promedio(self) -> float:
    if not self._calificaciones:
      return nan
    ponderadas = [c for c in self._calificaciones if isinstance(c, CalificacionPonderada)]
    if not ponderadas:
      return super().promedio # usa el promedio simple si no hay ponderadas
    # Promedio ponderado
    total_valor = sum(c.valor * c.peso for c in ponderadas)
    total_peso = sum(c.peso for c in ponderadas)
    return total_valor/total_peso if total_peso>0 else nan

  def __repr__(self):
    return f"EstudianteUniversitario(nombre = {self.nombre}, edad={self.edad},c_calif =len(self._calificaciones))"

#### c. `EstudianteBecado` (hereda de `EstudianteUniversitario`)

Requiere un promedio minimo para mantener una besa

In [43]:
class EstudianteBecado(EstudianteUniversitario):
  """
  Estudiante con beca que requiere un promedio minimo
  """
  def __init__(self, nombre: str, edad: int, promedio_minimo=8.0):
    super().__init__(nombre, edad)
    self.promedio_mninimo = promedio_minimo


  @property
  def beca_activa(self) -> bool:
    return self.promedio >= self.promedio_minimo

  def __repr__(self) :
    if self.beca_activa() :
      estado = "ACTIVA"
    else:
      estado = "SUSPENDIDA"
      return f"EstudianteBecado(nombre = {self.nombre},edad = {self.edad}. estado={estado})"